# GSPO: STEM Reasoning via Verifiable Rewards + Curriculum Learning

This notebook applies **GSPO** (Group Sequence Policy Optimization) with verifiable
rewards and **curriculum learning** to improve STEM reasoning quality.

**Training pipeline stage:** 1 of 4 (**GSPO (curriculum)** → RAFT++ → AdaSTaR → DPO)

**Target hardware:** Google Colab A100 40GB / 80GB

**Key features:**
- `importance_sampling_level="sequence"` — GSPO sequence-level importance ratios (Qwen3 training)
- `loss_type="sapo"` — SAPO smooth adaptive clipping (Qwen-native, arXiv 2511.20347)
- `beta=0.0` — No KL regularization (GSPO/DAPO standard)
- `epsilon=0.2`, `epsilon_high=0.28` — asymmetric Clip-Higher (DAPO/VAPO-aligned)
- `mask_truncated_completions=True` — exclude truncated generations from loss
- `steps_per_generation=16`, `gradient_accumulation_steps=4`
- G=16 completions per prompt (A100 40GB) / G=32 (A100 80GB)
- **SAPO loss (arXiv 2511.20347):** Temperature-controlled smooth clipping from Qwen team
- **ReDit reward dithering (arXiv 2506.18631):** Gaussian noise on binary rewards for 10x convergence
- **Dr. GRPO (arXiv 2503.20783):** Constant length normalization to prevent length bias
- **GDPO-style decoupled reward normalization:** correctness and format rewards
  normalized independently to prevent reward hacking (arXiv 2601.05242)
- **Curriculum learning (GRPO-LEAD):** Stage 1 trains easy+medium, Stage 2 all with difficulty reweighting
- **Difficulty-aware advantage reweighting:** hard=2×, easy=0.5× (arXiv 2504.09696)
- **Zero-variance masking:** skip groups where all G completions got identical reward (arXiv 2505.22257)
- Domain-specific reward functions: correct=1.0, wrong=0.0 (no negative penalties)
- Format reward (0.2 weight) for \boxed{} and step-by-step reasoning

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

**Note:** SFT stage was removed — Qwen3-4B-Instruct already has dialogue/instruction-following
abilities built-in, making SFT redundant. GSPO trains directly on top of the Instruct model.

**References:**
- [GSPO paper (arXiv 2507.18071)](https://arxiv.org/abs/2507.18071)
- [SAPO: Soft Adaptive Policy Optimization (arXiv 2511.20347)](https://arxiv.org/abs/2511.20347)
- [ReDit: Reward Dithering (arXiv 2506.18631)](https://arxiv.org/abs/2506.18631)
- [Dr. GRPO (arXiv 2503.20783)](https://arxiv.org/abs/2503.20783)
- [Qwen3 blog — GSPO section](https://qwenlm.github.io/blog/gspo/)
- [Qwen3 Technical Report (arXiv 2505.09388)](https://arxiv.org/pdf/2505.09388)
- [DAPO (arXiv 2503.14476)](https://arxiv.org/pdf/2503.14476) — Clip-Higher, Dynamic Sampling
- [VAPO (arXiv 2504.05118)](https://arxiv.org/abs/2504.05118) — epsilon=0.2/0.28 clipping bounds
- [GDPO (arXiv 2601.05242)](https://arxiv.org/abs/2601.05242) — decoupled multi-reward normalization
- [DRPO (arXiv 2510.04474)](https://arxiv.org/abs/2510.04474) — no negative reward penalties
- [GRPO-LEAD (arXiv 2504.09696)](https://arxiv.org/abs/2504.09696) — difficulty-aware curriculum RL
- [Revisiting GRPO (arXiv 2505.22257)](https://arxiv.org/abs/2505.22257) — zero-variance masking
- [DeepSeekMath (arXiv 2402.03300)](https://arxiv.org/pdf/2402.03300)
- [TRL GSPO docs](https://github.com/huggingface/trl/blob/main/docs/source/paper_index.md)

In [ ]:
# Disable gradient offloading BEFORE importing unsloth
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"
# Better CUDA error reporting — shows actual failing kernel instead of async error
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# Install dependencies — TRL >= 0.27.0 required for SAPO loss support
!pip install -q unsloth "trl>=0.27.0" peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy  # For verifiable reward functions

# Login to HuggingFace (needed for private SFT adapter)
from huggingface_hub import login
login()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# Add Drive root to sys.path (training/ is at MyDrive level)
# ============================================================
import sys, os

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py", "training/scripts/sort_curriculum.py"]:
    print(f"  {'OK' if os.path.exists(os.path.join(DRIVE_ROOT, s)) else 'MISSING'} {s}")

  MISSING training/scripts/stem_rewards.py
  MISSING training/scripts/verify_answers.py
  MISSING training/scripts/sort_curriculum.py


In [ ]:
# ============================================================                                                  # Configuration — GSPO-aligned (arXiv 2507.18071)                                                               # ============================================================
# Model — Instruct base gives dialogue abilities built-in + better RL starting point
# SFT stage removed: Instruct model already has instruction-following, SFT was redundant
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3_4b"

# ---- A100 GPU preset ----
A100_VRAM_GB = 80

# ---- GSPO core (arXiv 2507.18071) ----
IMPORTANCE_SAMPLING_LEVEL = "sequence"

# ---- Loss type ----
# Dr. GRPO: constant length normalization to prevent length bias (arXiv 2503.20783)
# Works with Unsloth's compiled loss (unlike SAPO which is blocked)
LOSS_TYPE = "dr_grpo"

# ---- Clipping bounds ----
# CRITICAL: sequence-level IS requires ε 2-3 orders of magnitude smaller
# than token-level. GSPO paper Section 5.1: ε=3e-4, ε_high=4e-4
# With token-level IS these would be 0.2/0.28 (DAPO/VAPO style),
# but sequence-level ratios have fundamentally different scale.
EPSILON = 3e-4              # GSPO paper (was 0.2 — root cause of loss spikes)
EPSILON_HIGH = 4e-4         # GSPO paper (was 0.28)

BETA = 0.0                  # No KL regularization (GSPO/DAPO standard)
MAX_PROMPT_LENGTH = 512
MAX_GRAD_NORM = 0.1         # Unsloth GSPO docs recommend 0.1 (was 1.0 — caused collapse at step 75)

# ---- Sequence length budget ----
# MAX_SEQ_LENGTH = model's RoPE/attention ceiling (passed to FastLanguageModel).
# Must be > MAX_PROMPT_LENGTH + MAX_COMPLETION to avoid Unsloth torch.gather crash
# when a prompt exceeds MAX_PROMPT_LENGTH before TRL truncation kicks in.
# With flash attention, memory scales with ACTUAL lengths, not the ceiling,
# so extra headroom is essentially free.
MAX_SEQ_LENGTH = 2048       # was 1280 — crashed at step 142 (input_ids=1857 > 1280)

# ---- Stage 1 optimizer: AdamW (already trained, adapter saved) ----
LEARNING_RATE = 1e-6        # Consensus: DeepSeek-Math, DAPO, VAPO, Dr. GRPO, TRL default
WEIGHT_DECAY = 0.1          # Prevents LoRA weight norm growth (DeepSeek-Math, VAPO)
ADAM_BETA2 = 0.99           # Faster second-moment adaptation (was 0.999 default — slow recovery)

# ---- Stage 2 optimizer: Lion (arXiv 2302.06675) ----
# Sign-based momentum: natural robustifier against heavy-tailed RL gradients.
# Only tracks momentum (1 state) → 33% less memory than AdamW (2 states).
# Lion requires 3-10x smaller LR and 3-10x larger weight_decay vs AdamW.
# Valid names: lion_8bit, lion_32bit, paged_lion_8bit, paged_lion_32bit
STAGE2_OPTIMIZER = "lion_8bit"          # bitsandbytes Lion 8-bit
STAGE2_LEARNING_RATE = 3e-7             # Lion: 3x smaller than AdamW (1e-6 / 3)
STAGE2_WEIGHT_DECAY = 0.3              # Lion: 3x larger (0.1 * 3)

# ---- ReDit: Reward Dithering (arXiv 2506.18631) ----
DITHERING_SIGMA = 0.05

# ---- A100 80GB VRAM settings ----
# Unsloth forces per_device_train_batch_size = num_generations.
# importance_sampling_level="sequence" requires ~2x forward pass memory.
# G=16 tested stable with grad_accum=2 on A100 80GB.
if A100_VRAM_GB >= 80:
    G = 16                       # GSPO paper uses 16
    BATCH_SIZE = 16              # Must equal G (Unsloth requirement)
    MAX_COMPLETION = 768         # 1024 OOMs with importance_sampling double pass
else:
    G = 8
    BATCH_SIZE = 8
    MAX_COMPLETION = 512

# LoRA for GSPO (fresh LoRA on top of Instruct base)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ---- Training (GSPO-aligned) ----
# steps_per_generation / grad_accum = optimizer updates per rollout
# 8 / 2 = 4 optimizer updates (matches GSPO paper)
# With ε=3e-4, gradient explosion is impossible — ratio clamped to [0.9997, 1.0004]
GRADIENT_ACCUMULATION_STEPS = 2     # was 4; effective batch = 16×2 = 32
STEPS_PER_GENERATION = 8            # was 16; 25 generation rounds for 200 steps
LOGGING_STEPS = 5
SAVE_STEPS = 25                     # was 100; recovery points every ~25 min

# GDPO-style decoupled reward weights (arXiv 2601.05242)
REWARD_WEIGHTS = [0.8, 0.2]        # [correctness, format]

# ---- Curriculum learning (GRPO-LEAD, arXiv 2504.09696) ----
CURRICULUM_CONFIG = {
    "stage1_steps": 200,
    "stage2_steps": 400,
    "difficulty_weights": {
        "easy": 0.5,
        "medium": 1.0,
        "hard": 2.0,
    },
}
TOTAL_STEPS = CURRICULUM_CONFIG["stage1_steps"] + CURRICULUM_CONFIG["stage2_steps"]

# Domains
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

effective_batch = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
print(f"Hardware: A100 {A100_VRAM_GB}GB")
print(f"Base model: {BASE_MODEL} (Instruct — no SFT stage)")
print(f"Loss type: {LOSS_TYPE}")
print(f"IS level: {IMPORTANCE_SAMPLING_LEVEL}, epsilon={EPSILON}, epsilon_high={EPSILON_HIGH}")
print(f"ReDit dithering: sigma={DITHERING_SIGMA}")
print(f"G={G} completions, max_completion={MAX_COMPLETION}, max_prompt={MAX_PROMPT_LENGTH}")
print(f"Max seq length: {MAX_SEQ_LENGTH} (model ceiling, headroom={MAX_SEQ_LENGTH - MAX_PROMPT_LENGTH - MAX_COMPLETION})")
print(f"Batch: {BATCH_SIZE} x grad_accum={GRADIENT_ACCUMULATION_STEPS} = {effective_batch} effective")
print(f"Steps per generation: {STEPS_PER_GENERATION} → {STEPS_PER_GENERATION // GRADIENT_ACCUMULATION_STEPS} optimizer updates/rollout")
print(f"Stage 1: optim=adamw_torch_fused, LR={LEARNING_RATE}, wd={WEIGHT_DECAY}, beta2={ADAM_BETA2}")
print(f"Stage 2: optim={STAGE2_OPTIMIZER}, LR={STAGE2_LEARNING_RATE}, wd={STAGE2_WEIGHT_DECAY} (Lion)")
print(f"max_grad_norm={MAX_GRAD_NORM}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA} (fresh)")
print(f"Curriculum: Stage 1={CURRICULUM_CONFIG['stage1_steps']} steps, Stage2={CURRICULUM_CONFIG['stage2_steps']} steps")
print(f"GDPO reward weights: correctness={REWARD_WEIGHTS[0]}, format={REWARD_WEIGHTS[1]}")
print(f"Save every {SAVE_STEPS} steps")

In [ ]:
# ============================================================
# Mount Google Drive
# ============================================================
import json
import os
import sys
from collections import Counter, defaultdict

# Try mounting Drive; fall back to local
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/gspo_qwen3_4b"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# Load RL problems: Drive JSONL → HF "rl" config → HF "gspo" fallback
# ============================================================

import json
from datasets import load_dataset
from pathlib import Path

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

def load_rl_problems():
    """Load RL dataset with fallback chain:
    1. Local JSONL from Google Drive (prepared by prepare_rl_dataset.py)
    2. HuggingFace "rl" config (future upload)
    3. HuggingFace "gspo" config (original, for backwards compat)
    """
    # Try Drive JSONL first
    if os.path.exists(RL_DATA_PATH):
        print(f"Loading RL data from Drive: {RL_DATA_PATH}")
        problems = []
        with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
            for line in f:
                problems.append(json.loads(line))
        print(f"Loaded {len(problems)} problems from Drive JSONL")
        return problems

    # Try HF "rl" config
    try:
        print("Trying HF dataset config 'rl'...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
        problems = [dict(r) for r in hf_ds["train"]]
        if "test" in hf_ds:
            problems += [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} problems from HF 'rl' config")
        return problems
    except Exception:
        pass

    # Fallback: HF "gspo" config
    print("Falling back to HF 'gspo' config...")
    hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
    problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
    print(f"Loaded {len(problems)} problems from HF 'gspo' config (fallback)")
    return problems


problems = load_rl_problems()

# Filter to verifiable problems only
verifiable_problems = [
    p for p in problems
    if p.get("type", "verifiable") == "verifiable"
    and p.get("answer_type", "numeric") != "conceptual"
]
conceptual_count = len(problems) - len(verifiable_problems)

print(f"\nTotal loaded: {len(problems)}")
print(f"  Verifiable (used for training): {len(verifiable_problems)}")
if conceptual_count > 0:
    print(f"  Conceptual (skipped — no verifiable reward signal): {conceptual_count}")

# Stats
from collections import Counter
domain_counts = Counter(p.get("domain", "unknown") for p in verifiable_problems)
source_counts = Counter(p.get("source", "unknown") for p in verifiable_problems)
type_counts = Counter(p.get("answer_type", "unknown") for p in verifiable_problems)

print(f"\nBy domain:")
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count}")
print(f"\nBy source:")
for source, count in sorted(source_counts.items()):
    print(f"  {source}: {count}")
print(f"\nBy answer type:")
for atype, count in sorted(type_counts.items()):
    print(f"  {atype}: {count}")

Loading RL data from Drive: /content/drive/MyDrive/training/data/rl_combined.jsonl
Loaded 14203 problems from Drive JSONL

Total loaded: 14203
  Verifiable (used for training): 14203

By domain:
  biology: 307
  chemistry: 162
  cs: 258
  math: 12931
  physics: 545

By source:
  current_filtered: 91
  gsm8k: 7400
  math_hendrycks: 4985
  olympiad_bench: 232
  rummlu: 1495

By answer type:
  latex_boxed: 4985
  mc_letter: 1495
  numeric: 7491
  numeric_with_unit: 232


In [ ]:
# ============================================================
# Held-out evaluation split (10% stratified by domain)
# ============================================================
import random

EVAL_FRACTION = 0.10
random.seed(42)

_domain_buckets = {}
for p in verifiable_problems:
    d = p.get("domain", "math")
    _domain_buckets.setdefault(d, []).append(p)

train_problems = []
eval_problems = []
for domain, probs in _domain_buckets.items():
    random.shuffle(probs)
    n_eval = max(1, int(len(probs) * EVAL_FRACTION))
    eval_problems.extend(probs[:n_eval])
    train_problems.extend(probs[n_eval:])

print(f"\nHeld-out split (seed=42):")
print(f"  Train: {len(train_problems)}")
print(f"  Eval:  {len(eval_problems)}")
eval_domains = Counter(p.get("domain") for p in eval_problems)
for d, c in sorted(eval_domains.items()):
    print(f"    {d}: {c}")

# Use train_problems for curriculum splits below
verifiable_problems = train_problems

In [ ]:
# ============================================================
# Reward functions: import from stem_rewards.py (GDPO-compatible)
# Fallback: inline definitions if import fails (Colab path issues)
# ============================================================

# Try importing GDPO reward functions from stem_rewards.py
_stem_rewards_imported = False
try:
    from training.scripts.stem_rewards import make_gdpo_reward_fns, make_gdpo_correctness_fn, make_gdpo_format_fn
    _stem_rewards_imported = True
    print("Imported GDPO reward functions from training.scripts.stem_rewards")
except ImportError:
    try:
        import sys
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.stem_rewards import make_gdpo_reward_fns, make_gdpo_correctness_fn, make_gdpo_format_fn
        _stem_rewards_imported = True
        print("Imported GDPO reward functions (via Drive path)")
    except ImportError:
        print("WARNING: Could not import stem_rewards, defining fallback reward functions")

if not _stem_rewards_imported:
    # ---- Fallback reward functions (same logic as stem_rewards.py) ----
    import sympy
    import re

    def _extract_boxed_answer(text):
        """Extract answer from \\boxed{...} or final numeric answer."""
        boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
        if boxed:
            return boxed[-1].strip()
        numbers = re.findall(r"[-+]?\d*\.?\d+", text)
        return numbers[-1] if numbers else ""

    def _verify_answer(completion, answer, domain):
        """Simple verification: returns 1.0 if correct, 0.0 otherwise."""
        extracted = _extract_boxed_answer(completion)
        if not extracted or not answer:
            return 0.0
        if domain == "math":
            try:
                pred = sympy.sympify(extracted)
                gold = sympy.sympify(answer)
                if sympy.simplify(pred - gold) == 0:
                    return 1.0
                return 0.0
            except (sympy.SympifyError, TypeError, ValueError):
                return 1.0 if extracted.strip() == str(answer).strip() else 0.0
        elif domain == "physics":
            try:
                pred_num = float(re.findall(r"[-+]?\d*\.?\d+", extracted)[0])
                gold_num = float(re.findall(r"[-+]?\d*\.?\d+", str(answer))[0])
                return 1.0 if abs(pred_num - gold_num) / max(abs(gold_num), 1e-10) < 0.05 else 0.0
            except (ValueError, IndexError):
                return 1.0 if extracted.strip() == str(answer).strip() else 0.0
        else:
            return 1.0 if extracted.strip().lower() == str(answer).strip().lower() else 0.0

print("Reward functions ready (GDPO-compatible, no negative penalties)")

Imported GDPO reward functions from training.scripts.stem_rewards
Reward functions ready (GDPO-compatible, no negative penalties)


In [ ]:
# ============================================================
# Classify problem difficulty + build curriculum splits
# ============================================================
from collections import Counter

try:
    from training.scripts.sort_curriculum import classify_difficulty
    print("Imported classify_difficulty from training.scripts.sort_curriculum")
except ImportError:
    def classify_difficulty(example):
        """Fallback: classify by word count heuristic."""
        text = example.get("answer", "") + " " + example.get("prompt", "")
        word_count = len(text.split())
        if word_count < 50: return "easy"
        elif word_count > 150: return "hard"
        return "medium"
    print("Using fallback classify_difficulty")

for p in verifiable_problems:
    p["difficulty"] = classify_difficulty(p)

difficulty_dist = Counter(p["difficulty"] for p in verifiable_problems)
print(f"Difficulty distribution: {dict(difficulty_dist)}")

easy_medium_problems = [p for p in verifiable_problems if p["difficulty"] in ("easy", "medium")]
hard_problems = [p for p in verifiable_problems if p["difficulty"] == "hard"]
print(f"Stage 1 (easy+medium): {len(easy_medium_problems)} problems")
print(f"Stage 2 adds hard: {len(hard_problems)} problems")
print(f"Total: {len(verifiable_problems)} problems")

Imported classify_difficulty from training.scripts.sort_curriculum
Difficulty distribution: {'easy': 5673, 'hard': 3584, 'medium': 4946}
Stage 1 (easy+medium): 10619 problems
Stage 2 adds hard: 3584 problems
Total: 14203 problems


In [ ]:
# ============================================================
# Load base Instruct model + fresh LoRA for GSPO
# ============================================================
import torch
import json
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import random
import numpy as np

# ---- Load Instruct base + fresh LoRA ----
# No SFT adapter: Qwen3-4B-Instruct already has instruction-following,
# so we apply a fresh LoRA directly for GSPO RL training.
# NOTE: max_seq_length is the MODEL's internal ceiling (RoPE, attention buffers).
# It must exceed any possible prompt+completion length to prevent torch.gather
# shape mismatch in Unsloth's compiled GRPO loss. With flash attention,
# memory depends on actual lengths, not the ceiling — so headroom is free.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,      # 2048 (was 1280 — crashed on 1857-token sequence)
    load_in_4bit=True,
    dtype=torch.bfloat16,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Loaded {BASE_MODEL} + fresh LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  max_seq_length={MAX_SEQ_LENGTH} (headroom={MAX_SEQ_LENGTH - MAX_PROMPT_LENGTH - MAX_COMPLETION} over prompt+completion)")

if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

# ============================================================
# Type-aware system prompts
# ============================================================
SYSTEM_PROMPT_CALC = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."
SYSTEM_PROMPT_MC = "Проанализируй задачу и выбери правильный ответ (A, B, C или D)."

# Legacy alias for curriculum/reward code
SYSTEM_PROMPT = SYSTEM_PROMPT_CALC

def get_system_prompt(problem):
    """Return appropriate system prompt based on answer_type."""
    if problem.get("answer_type") == "mc_letter":
        return SYSTEM_PROMPT_MC
    return SYSTEM_PROMPT_CALC


# ============================================================
# GDPO-style reward functions + difficulty-aware wrappers
# ============================================================

def format_problems_as_dataset(problem_list):
    """Format problems into a Dataset for GRPOTrainer.
    Routes each problem to the correct system prompt by answer_type.
    """
    formatted = []
    for p in problem_list:
        messages = [
            {"role": "system", "content": get_system_prompt(p)},
            {"role": "user", "content": p["prompt"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        formatted.append({"prompt": prompt})
    return Dataset.from_list(formatted)


# Build prompt-to-problem lookup (shared by reward functions)
# Registers each problem with its type-appropriate system prompt
_prompt_to_problem = {}
for p in verifiable_problems:
    messages = [
        {"role": "system", "content": get_system_prompt(p)},
        {"role": "user", "content": p["prompt"]},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    _prompt_to_problem[formatted.strip()] = p


if _stem_rewards_imported:
    # Use proper GDPO functions from stem_rewards.py
    # These now handle MC/calc routing + ReDit dithering internally
    _gdpo_fns = make_gdpo_reward_fns(
        verifiable_problems, tokenizer, SYSTEM_PROMPT_CALC,
        dithering_sigma=DITHERING_SIGMA,  # ReDit (arXiv 2506.18631)
    )
    _base_correctness_fn = _gdpo_fns[0]
    _base_format_fn = _gdpo_fns[1]
    print(f"  ReDit dithering: sigma={DITHERING_SIGMA}")
else:
    # Fallback: build correctness fn using inline _verify_answer
    def _base_correctness_fn(completions, prompts=None, **kwargs):
        if prompts is None:
            prompts = [""] * len(completions)
        rewards = []
        for prompt_text, completion_text in zip(prompts, completions):
            problem = _prompt_to_problem.get(prompt_text.strip())
            if problem is None:
                rewards.append(0.0)
                continue
            domain = problem.get("domain", "math")
            answer = problem.get("answer", "")
            answer_type = problem.get("answer_type", "numeric")

            if answer_type == "mc_letter":
                import re
                mc_match = re.search(r'\b([A-DА-Г])\b', _extract_boxed_answer(completion_text) or completion_text[-20:])
                rewards.append(1.0 if mc_match and mc_match.group(1).upper() == answer.upper() else 0.0)
            else:
                rewards.append(_verify_answer(completion_text, answer, domain))

        # Fallback ReDit dithering
        if DITHERING_SIGMA > 0:
            noise = np.random.normal(0.0, DITHERING_SIGMA, size=len(rewards))
            rewards = [r + n for r, n in zip(rewards, noise)]

        return rewards

    def _base_format_fn(completions, prompts=None, **kwargs):
        rewards = []
        for i, text in enumerate(completions):
            text = text if isinstance(text, str) else str(text)
            answer_type = "numeric"
            if prompts and i < len(prompts):
                problem = _prompt_to_problem.get(prompts[i].strip())
                if problem:
                    answer_type = problem.get("answer_type", "numeric")

            score = 0.0
            if answer_type == "mc_letter":
                import re
                if re.search(r'(?:answer|ответ)\s*[:=]\s*[A-DА-Г]', text, re.IGNORECASE):
                    score += 0.5
                step_markers = ["because", "therefore", "потому что", "так как", "следовательно"]
                if any(m in text.lower() for m in step_markers):
                    score += 0.3
            else:
                if "\\boxed{" in text:
                    score += 0.5
                step_markers = ["step", "therefore", "thus", "hence", "because",
                                "шаг", "следовательно", "значит", "потому что",
                                "так как", "далее", "подставим", "найдём"]
                if any(m in text.lower() for m in step_markers):
                    score += 0.3
            word_count = len(text.split())
            if 50 < word_count < 800:
                score += 0.2
            rewards.append(min(1.0, score))
        return rewards


# ---- Difficulty-aware correctness reward (GRPO-LEAD) ----
DIFFICULTY_WEIGHTS = CURRICULUM_CONFIG["difficulty_weights"]

def difficulty_weighted_correctness_fn(completions, prompts=None, **kwargs):
    """Correctness reward × difficulty weight (GRPO-LEAD reweighting)."""
    base_rewards = _base_correctness_fn(completions, prompts=prompts, **kwargs)
    if prompts is None:
        return base_rewards
    weighted = []
    for reward, prompt_text in zip(base_rewards, prompts):
        problem = _prompt_to_problem.get(prompt_text.strip())
        if problem is not None:
            diff = problem.get("difficulty", "medium")
            weight = DIFFICULTY_WEIGHTS.get(diff, 1.0)
            weighted.append(reward * weight)
        else:
            weighted.append(reward)
    return weighted


# ---- Zero-variance masking (arXiv 2505.22257) ----
_zero_var_groups_masked = 0
_total_groups_seen = 0

def zero_variance_masked_correctness_fn(completions, prompts=None, **kwargs):
    """Correctness reward with zero-variance group masking.

    If all G completions for a prompt get the same reward (std < 1e-6),
    returns NaN for the entire group so TRL skips it.
    FIX: Use threshold instead of == 0, since ReDit dithering adds noise
    that makes exact zero std impossible but near-zero std still uninformative.
    """
    global _zero_var_groups_masked, _total_groups_seen

    rewards = difficulty_weighted_correctness_fn(completions, prompts=prompts, **kwargs)

    if prompts is not None:
        prompt_groups = defaultdict(list)
        for i, prompt_text in enumerate(prompts):
            prompt_groups[prompt_text.strip()].append(i)

        for prompt_text, indices in prompt_groups.items():
            _total_groups_seen += 1
            group_rewards = [rewards[i] for i in indices]
            if len(group_rewards) > 1 and np.std(group_rewards) < 1e-6:
                _zero_var_groups_masked += 1
                for i in indices:
                    rewards[i] = float('nan')

    return rewards


# Stats
mc_count = sum(1 for p in verifiable_problems if p.get("answer_type") == "mc_letter")
calc_count = len(verifiable_problems) - mc_count
print(f"\nModel and GDPO reward functions ready for curriculum GSPO training")
print(f"  Loss type: {LOSS_TYPE}")
print(f"  ReDit dithering: sigma={DITHERING_SIGMA}")
print(f"  Calc problems (\\boxed{{}}): {calc_count}")
print(f"  MC problems (A/B/C/D): {mc_count}")
print(f"  Difficulty weights: {DIFFICULTY_WEIGHTS}")
print(f"  Zero-variance masking: enabled (threshold < 1e-6)")

In [ ]:
# ============================================================
# Unsloth/Qwen3 compatibility patches (ALL IDEMPOTENT)
# ============================================================
import torch

# CORRECT vocab size: includes special tokens (pad, eos, im_start, etc.)
# tokenizer.vocab_size = 151643 (base only, WRONG for clamping!)
# model embedding includes all tokens
_vocab_size = model.get_input_embeddings().num_embeddings
print(f"Using embedding vocab_size={_vocab_size} (not tokenizer.vocab_size={tokenizer.vocab_size})")

# ---- Patch 1: has_images ----
try:
    _gen_method = GRPOTrainer._generate_and_score_completions
    if "has_images" not in _gen_method.__globals__ or _gen_method.__globals__.get("has_images") is None:
        _gen_method.__globals__["has_images"] = False
        print("[patch 1/4] has_images=False injected")
    else:
        print(f"[patch 1/4] has_images OK")
except AttributeError:
    print("[patch 1/4] not needed")

# ---- Patch 2: batch_decode safety net ----
_P2 = "_original_batch_decode_unpatched"
if not hasattr(tokenizer, _P2):
    _orig_batch_decode = tokenizer.batch_decode

    def _safe_batch_decode(sequences, skip_special_tokens=False, **kwargs):
        sanitized = []
        for seq in sequences:
            if isinstance(seq, torch.Tensor):
                seq = seq.clamp(0, _vocab_size - 1).tolist()
            elif isinstance(seq, list):
                seq = [max(0, min(t, _vocab_size - 1)) for t in seq]
            sanitized.append(seq)
        return _orig_batch_decode(sanitized, skip_special_tokens=skip_special_tokens, **kwargs)

    tokenizer.batch_decode = _safe_batch_decode
    tokenizer._original_batch_decode_unpatched = _orig_batch_decode
    print(f"[patch 2/4] batch_decode clamped to [0, {_vocab_size})")
else:
    print("[patch 2/4] batch_decode OK")

# ---- Patch 3: embed_tokens safety net ----
_embed_layer = model.get_input_embeddings()
_P3 = "_original_embed_forward_unpatched"
if not hasattr(_embed_layer, _P3):
    _orig_embed_fwd = _embed_layer.forward

    def _safe_embed_forward(input_tensor):
        return _orig_embed_fwd(input_tensor.clamp(0, _vocab_size - 1))

    _embed_layer.forward = _safe_embed_forward
    _embed_layer._original_embed_forward_unpatched = _orig_embed_fwd
    print(f"[patch 3/4] embed_tokens clamped to [0, {_vocab_size})")
else:
    print("[patch 3/4] embed_tokens OK")

# ---- Patch 4: generation output safety net ----
_P4 = "_original_gen_score_unpatched"
_pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

if not hasattr(GRPOTrainer, _P4):
    _orig_gen_score = GRPOTrainer._generate_and_score_completions
    _ID_KEYS = {"input_ids", "prompt_ids", "completion_ids", "labels",
                "prompt_completion_ids", "old_input_ids"}

    def _safe_gen_score(self, inputs):
        result = GRPOTrainer._original_gen_score_unpatched(self, inputs)
        if isinstance(result, dict):
            for key, val in result.items():
                if key in _ID_KEYS and isinstance(val, torch.Tensor):
                    bad = (val < 0) | (val >= _vocab_size)
                    if bad.any():
                        # Replace with pad_token_id (not 0!) so completion_mask
                        # properly ignores these positions in loss computation
                        result[key] = torch.where(bad, _pad_id, val)
        return result

    GRPOTrainer._original_gen_score_unpatched = _orig_gen_score
    GRPOTrainer._generate_and_score_completions = _safe_gen_score
    print(f"[patch 4/4] generation: invalid IDs → pad_token_id={_pad_id}")
else:
    print("[patch 4/4] generation OK")

print(f"\n[info] pad={tokenizer.pad_token_id}, eos={tokenizer.eos_token_id}, bos={tokenizer.bos_token_id}")
print("All patches applied.")

Using embedding vocab_size=151936 (not tokenizer.vocab_size=151643)
[patch 1/4] has_images=False injected
[patch 2/4] batch_decode clamped to [0, 151936)
[patch 3/4] embed_tokens clamped to [0, 151936)
[patch 4/4] generation: invalid IDs → pad_token_id=151654

[info] pad=151654, eos=151645, bos=None
All patches applied.


In [ ]:
print(f"tokenizer.vocab_size     = {tokenizer.vocab_size}")
print(f"len(tokenizer)           = {len(tokenizer)}")
print(f"model.config.vocab_size  = {model.config.vocab_size}")
print(f"embed num_embeddings     = {model.get_input_embeddings().num_embeddings}")

tokenizer.vocab_size     = 151643
len(tokenizer)           = 151669
model.config.vocab_size  = 151936
embed num_embeddings     = 151936


In [ ]:
# ============================================================
# Baseline evaluation: Base Instruct model BEFORE GSPO training
# ============================================================
# This measures the starting point so we can quantify GSPO's improvement.
# Results are saved as stage_base_*.json and appended to summary.csv.

import sys
sys.path.insert(0, "/content/drive/MyDrive/MITS")

from training.scripts.evaluate_stage import (
    load_eval_dataset, evaluate_with_model, save_report,
    append_summary_csv, print_comparison,
)

EVAL_PATH = "/content/drive/MyDrive/MITS/training/data/eval_benchmark.jsonl"
REPORT_DIR = "/content/drive/MyDrive/MITS/evaluation/reports"
SUMMARY_CSV = os.path.join(REPORT_DIR, "summary.csv")

eval_problems = load_eval_dataset(EVAL_PATH)
print(f"\nEvaluating BASE model ({BASE_MODEL}) before training...")
base_results = evaluate_with_model(model, tokenizer, eval_problems)

save_report(base_results, "base", "colab", BASE_MODEL, REPORT_DIR)
append_summary_csv(base_results, "base", SUMMARY_CSV)
print_comparison(base_results, "base")

# Store baseline for post-training comparison
_baseline_results = base_results
print("\nBaseline saved. Proceeding to GSPO training...")

In [ ]:
# ============================================================
# Training: two-stage curriculum with GDPO-style decoupled rewards
# Stage 1: easy+medium only (warm-up, AdamW, stabilize policy)
# Stage 2: all tiers, Lion optimizer, difficulty reweighting
# Optimizations: Dr. GRPO loss, ReDit dithering, Clip-Higher
# (with Colab disconnect recovery)
# ============================================================

def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect.

    Validates that the checkpoint's LoRA rank matches the current LORA_R
    config to prevent size mismatch errors on resume.
    """
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    # Check from newest to oldest — first compatible one wins
    for ckpt_name in sorted(checkpoints, key=lambda x: int(x.split("-")[1]), reverse=True):
        path = os.path.join(output_dir, ckpt_name)
        adapter_config_path = os.path.join(path, "adapter_config.json")
        if os.path.exists(adapter_config_path):
            with open(adapter_config_path, "r") as f:
                adapter_cfg = json.load(f)
            ckpt_r = adapter_cfg.get("r", None)
            if ckpt_r is not None and ckpt_r != LORA_R:
                print(f"  SKIP {ckpt_name}: LoRA r={ckpt_r} != current LORA_R={LORA_R}")
                continue
        print(f"  Found compatible checkpoint: {path}")
        return path
    print(f"  No compatible checkpoints found in {output_dir} (all have mismatched LoRA rank)")
    return None


def make_gspo_config(output_dir, max_steps, warmup_ratio=0.05, **overrides):
    """Create GRPOConfig for a training stage.

    Unsloth wraps GRPOConfig with a fixed __init__ that blocks newer TRL
    params. We work around this by:
    1. Passing only params accepted by Unsloth's __init__
    2. Injecting the rest as attributes AFTER construction
    TRL's trainer reads config attrs at runtime, so injected params work.

    **overrides: stage-specific param overrides (e.g. optim, learning_rate
    for Lion in Stage 2). Applied on top of defaults.
    """
    import inspect

    # All desired params (defaults = Stage 1 / AdamW)
    all_kwargs = dict(
        output_dir=output_dir,
        max_steps=max_steps,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=warmup_ratio,
        num_generations=G,
        max_completion_length=MAX_COMPLETION,
        max_prompt_length=MAX_PROMPT_LENGTH,
        # Loss type: "sapo", "dr_grpo", "grpo", "dapo"
        loss_type=LOSS_TYPE,
        beta=BETA,
        # Clip-Higher (DAPO/VAPO)
        epsilon=EPSILON,
        epsilon_high=EPSILON_HIGH,
        # GSPO (arXiv 2507.18071)
        importance_sampling_level=IMPORTANCE_SAMPLING_LEVEL,
        steps_per_generation=STEPS_PER_GENERATION,
        mask_truncated_completions=True,
        # GDPO reward weights (arXiv 2601.05242)
        reward_weights=REWARD_WEIGHTS,
        # Standard training params
        bf16=True,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        optim="adamw_torch_fused",
        max_grad_norm=MAX_GRAD_NORM,
        # Optimizer stability (post-audit: DeepSeek-Math, VAPO best practices)
        weight_decay=WEIGHT_DECAY,
        adam_beta2=ADAM_BETA2,
        temperature=0.9,
        seed=42,
        report_to="none",
    )

    # SAPO temperature params (arXiv 2511.20347)
    if LOSS_TYPE == "sapo":
        all_kwargs["sapo_temperature_pos"] = SAPO_TEMPERATURE_POS
        all_kwargs["sapo_temperature_neg"] = SAPO_TEMPERATURE_NEG

    # Apply stage-specific overrides (e.g. Lion optimizer for Stage 2)
    if overrides:
        all_kwargs.update(overrides)
        print(f"  Stage overrides applied: {list(overrides.keys())}")

    # Split: params accepted by __init__ vs post-init injection
    sig = inspect.signature(GRPOConfig.__init__)
    valid_init = set(sig.parameters.keys())

    init_kwargs = {}
    post_kwargs = {}
    for k, v in all_kwargs.items():
        if k in valid_init:
            init_kwargs[k] = v
        else:
            post_kwargs[k] = v

    config = GRPOConfig(**init_kwargs)

    # Inject remaining params as attributes
    for k, v in post_kwargs.items():
        setattr(config, k, v)

    if post_kwargs:
        print(f"  GRPOConfig: injected post-init: {list(post_kwargs.keys())}")

    return config


stage_metrics = {}
os.makedirs(OUTPUT_DIR, exist_ok=True)
stage1_dir = os.path.join(OUTPUT_DIR, "stage1")
stage2_dir = os.path.join(OUTPUT_DIR, "stage2")
stage1_adapter_path = os.path.join(OUTPUT_DIR, "stage1_adapter")

# ========================
# Stage 1: Easy + Medium problems (warm-up, AdamW)
# ========================
stage1_steps = CURRICULUM_CONFIG["stage1_steps"]

# Check if stage1 adapter exists AND has matching LoRA rank
_stage1_done = False
_s1_adapter_file = os.path.join(stage1_adapter_path, "adapter_model.safetensors")
_s1_config_file = os.path.join(stage1_adapter_path, "adapter_config.json")
if os.path.exists(_s1_adapter_file):
    if os.path.exists(_s1_config_file):
        with open(_s1_config_file, "r") as f:
            _s1_cfg = json.load(f)
        if _s1_cfg.get("r") == LORA_R:
            _stage1_done = True
        else:
            print(f"  Stage 1 adapter has r={_s1_cfg.get('r')}, current LORA_R={LORA_R} — retraining")
    else:
        # No config file — assume compatible (legacy checkpoint)
        _stage1_done = True

if _stage1_done:
    print(f"\n{'='*60}")
    print(f"STAGE 1: SKIPPED (adapter found at {stage1_adapter_path})")
    print(f"{'='*60}")
    from safetensors.torch import load_file as _load_s1
    model.load_state_dict(
        _load_s1(os.path.join(stage1_adapter_path, "adapter_model.safetensors")),
        strict=False,
    )
    print("  Loaded stage 1 adapter weights")
else:
    print(f"\n{'='*60}")
    print(f"STAGE 1: Easy + Medium ({stage1_steps} steps) — AdamW")
    print(f"{'='*60}")
    print(f"  Problems: {len(easy_medium_problems)} (easy+medium only)")
    print(f"  Optimizer: adamw_torch_fused, LR={LEARNING_RATE}")
    print(f"  Loss: {LOSS_TYPE}")
    print(f"  ReDit dithering: sigma={DITHERING_SIGMA}")
    print(f"  GDPO reward_funcs: [correctness (dithered + difficulty-weighted + zero-var masked), format]")
    print(f"  Reward weights: {REWARD_WEIGHTS}")

    stage1_ds = format_problems_as_dataset(easy_medium_problems)
    stage1_config = make_gspo_config(
        output_dir=stage1_dir,
        max_steps=stage1_steps,
        warmup_ratio=0.1,
    )

    _zero_var_groups_masked = 0
    _total_groups_seen = 0

    trainer_s1 = GRPOTrainer(
        model=model,
        args=stage1_config,
        train_dataset=stage1_ds,
        reward_funcs=[zero_variance_masked_correctness_fn, _base_format_fn],
        processing_class=tokenizer,
    )

    stage1_resume = find_latest_checkpoint(stage1_dir)
    print("Starting Stage 1 training...")
    result_s1 = trainer_s1.train(resume_from_checkpoint=stage1_resume)

    stage_metrics["stage1"] = {
        "steps": stage1_steps,
        "final_loss": result_s1.training_loss,
        "metrics": result_s1.metrics,
        "problems": len(easy_medium_problems),
        "tiers": "easy+medium",
        "optimizer": "adamw_torch_fused",
        "zero_var_groups_masked": _zero_var_groups_masked,
        "total_groups_seen": _total_groups_seen,
    }
    print(f"\nStage 1 complete! Loss: {result_s1.training_loss:.4f}")
    print(f"  Zero-variance groups masked: {_zero_var_groups_masked}/{_total_groups_seen}")

    model.save_pretrained(stage1_adapter_path)
    tokenizer.save_pretrained(stage1_adapter_path)
    print(f"  Stage 1 adapter saved to {stage1_adapter_path}")

    del trainer_s1
    torch.cuda.empty_cache()


# ========================
# Stage 2: All tiers with difficulty reweighting + Lion optimizer
# Lion (arXiv 2302.06675): sign-based momentum, robust to noisy RL gradients.
# Hyperparams adjusted per Lion paper: LR 3x smaller, weight_decay 3x larger.
# ========================
stage2_steps = CURRICULUM_CONFIG["stage2_steps"]
print(f"\n{'='*60}")
print(f"STAGE 2: All Tiers + Lion Optimizer ({stage2_steps} steps)")
print(f"{'='*60}")
print(f"  Problems: {len(verifiable_problems)} (all tiers)")
print(f"  Difficulty weights: {DIFFICULTY_WEIGHTS}")
print(f"  Optimizer: {STAGE2_OPTIMIZER} (sign-based momentum)")
print(f"  LR={STAGE2_LEARNING_RATE} (3x smaller for Lion), wd={STAGE2_WEIGHT_DECAY} (3x larger)")
print(f"  max_grad_norm={MAX_GRAD_NORM}")
print(f"  GDPO reward_funcs: [correctness (dithered + difficulty-weighted + zero-var masked), format]")

stage2_ds = format_problems_as_dataset(verifiable_problems)
stage2_config = make_gspo_config(
    output_dir=stage2_dir,
    max_steps=stage2_steps,
    warmup_ratio=0.08,              # longer warmup for hard problems (8% of 400 = 32 steps)
    # Lion optimizer overrides
    optim=STAGE2_OPTIMIZER,
    learning_rate=STAGE2_LEARNING_RATE,
    weight_decay=STAGE2_WEIGHT_DECAY,
)

_zero_var_groups_masked = 0
_total_groups_seen = 0

trainer_s2 = GRPOTrainer(
    model=model,
    args=stage2_config,
    train_dataset=stage2_ds,
    reward_funcs=[zero_variance_masked_correctness_fn, _base_format_fn],
    processing_class=tokenizer,
)

stage2_resume = find_latest_checkpoint(stage2_dir)
print("Starting Stage 2 training...")
result_s2 = trainer_s2.train(resume_from_checkpoint=stage2_resume)

stage_metrics["stage2"] = {
    "steps": stage2_steps,
    "final_loss": result_s2.training_loss,
    "metrics": result_s2.metrics,
    "problems": len(verifiable_problems),
    "tiers": "all (easy+medium+hard)",
    "optimizer": STAGE2_OPTIMIZER,
    "learning_rate": STAGE2_LEARNING_RATE,
    "weight_decay": STAGE2_WEIGHT_DECAY,
    "difficulty_weights": DIFFICULTY_WEIGHTS,
    "zero_var_groups_masked": _zero_var_groups_masked,
    "total_groups_seen": _total_groups_seen,
}
print(f"\nStage 2 complete! Loss: {result_s2.training_loss:.4f}")
print(f"  Zero-variance groups masked: {_zero_var_groups_masked}/{_total_groups_seen}")

trainer_s2.save_model(os.path.join(OUTPUT_DIR, "final"))
print(f"\nCurriculum GSPO training complete!")
if "stage1" in stage_metrics:
    print(f"  Stage 1 loss: {stage_metrics['stage1']['final_loss']:.4f} (AdamW)")
print(f"  Stage 2 loss: {result_s2.training_loss:.4f} (Lion)")

training_log = {
    "total_steps": TOTAL_STEPS,
    "stages": stage_metrics,
}

In [ ]:
# ============================================================
# Save final adapter + training config
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final GSPO adapter saved to {final_adapter_path}")

config_to_save = {
    "stage": "gspo",
    "pipeline": "GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "1 of 4",
    "base_model": BASE_MODEL,
    "note": "No SFT stage — Instruct model used directly",
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "importance_sampling_level": IMPORTANCE_SAMPLING_LEVEL,
    "loss_type": LOSS_TYPE,
    "beta": BETA,
    "epsilon": EPSILON,
    "epsilon_high": EPSILON_HIGH,
    "max_grad_norm": MAX_GRAD_NORM,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "max_completion_length": MAX_COMPLETION,
    # Stage 1: AdamW
    "stage1_optimizer": "adamw_torch_fused",
    "stage1_learning_rate": LEARNING_RATE,
    "stage1_weight_decay": WEIGHT_DECAY,
    "stage1_adam_beta2": ADAM_BETA2,
    # Stage 2: Lion (arXiv 2302.06675)
    "stage2_optimizer": STAGE2_OPTIMIZER,
    "stage2_learning_rate": STAGE2_LEARNING_RATE,
    "stage2_weight_decay": STAGE2_WEIGHT_DECAY,
    "stage2_note": "Lion sign-based momentum — robust to noisy RL gradients",
    # Common
    "steps_per_generation": STEPS_PER_GENERATION,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "mask_truncated_completions": True,
    "G": G,
    "max_completion": MAX_COMPLETION,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "total_steps": TOTAL_STEPS,
    "total_problems": len(verifiable_problems),
    # ReDit (arXiv 2506.18631)
    "dithering_sigma": DITHERING_SIGMA,
    # Curriculum config
    "curriculum_config": CURRICULUM_CONFIG,
    "difficulty_distribution": dict(difficulty_dist),
    # GDPO
    "reward_approach": "GDPO decoupled (arXiv 2601.05242)",
    "reward_weights": REWARD_WEIGHTS,
    "reward_scale": "correct=1.0, wrong=0.0 (no negative penalties)",
    "difficulty_weighting": "GRPO-LEAD (easy=0.5, medium=1.0, hard=2.0)",
    "zero_variance_masking": True,
    # Stage metrics
    "stage_metrics": stage_metrics,
    "references": [
        "GSPO arXiv:2507.18071",
        "Lion arXiv:2302.06675",
        "ReDit arXiv:2506.18631",
        "Dr. GRPO arXiv:2503.20783",
        "DAPO arXiv:2503.14476",
        "VAPO arXiv:2504.05118",
        "GDPO arXiv:2601.05242",
        "DRPO arXiv:2510.04474",
        "GRPO-LEAD arXiv:2504.09696",
        "Revisiting GRPO arXiv:2505.22257",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config saved to {config_path}")

# Optional: push to HF
PUSH_TO_HUB = True
HF_REPO_ID = "Siesher/mits-qwen3-4b-gspo"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! GSPO adapter ready for RAFT++ (next stage).")

In [ ]:
# ============================================================
# Post-Training Evaluation on MITS Benchmark
# ============================================================
import sys
sys.path.insert(0, "/content/drive/MyDrive/MITS")

from training.scripts.evaluate_stage import (
    load_eval_dataset, evaluate_with_model, save_report,
    append_summary_csv, print_comparison,
)

STAGE = "gspo"
EVAL_PATH = "/content/drive/MyDrive/MITS/training/data/eval_benchmark.jsonl"
REPORT_DIR = "/content/drive/MyDrive/MITS/evaluation/reports"
SUMMARY_CSV = os.path.join(REPORT_DIR, "summary.csv")

# Use in-memory baseline if available, otherwise load from file
import json, glob
baseline = _baseline_results if "_baseline_results" in dir() else None
if baseline is None:
    base_reports = sorted(glob.glob(os.path.join(REPORT_DIR, "stage_base_*.json")))
    if base_reports:
        with open(base_reports[-1], encoding="utf-8") as f:
            baseline = json.load(f).get("results")

eval_problems = load_eval_dataset(EVAL_PATH)
results = evaluate_with_model(model, tokenizer, eval_problems)

save_report(results, STAGE, "colab", HF_REPO_ID, REPORT_DIR)
append_summary_csv(results, STAGE, SUMMARY_CSV)
print_comparison(results, STAGE, baseline)

In [ ]:
# ============================================================
# Export to GGUF for local Ollama deployment
# ============================================================
# Unsloth merges LoRA weights into base model and quantizes to GGUF.
# Q8_0 recommended for 16GB RAM (~4GB model + ~1GB KV cache).
# Q4_K_M available if RAM is tight (~2.5GB model).

GGUF_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "gguf")
QUANTIZATION = "q8_0"  # Options: q4_k_m, q5_k_m, q8_0

print(f"Exporting GGUF ({QUANTIZATION}) from final adapter...")
print(f"  Adapter: {os.path.join(OUTPUT_DIR, 'final_adapter')}")
print(f"  Output:  {GGUF_OUTPUT_DIR}")

# Reload adapter if needed (model should still be in memory from training)
model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,
    quantization_method=QUANTIZATION,
)

# Find and report the exported file
gguf_files = [f for f in os.listdir(GGUF_OUTPUT_DIR) if f.endswith(".gguf")]
if gguf_files:
    gguf_path = os.path.join(GGUF_OUTPUT_DIR, gguf_files[0])
    size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
    print(f"\n✅ GGUF exported: {gguf_files[0]} ({size_mb:.0f} MB)")
    print(f"   Full path: {gguf_path}")
    print(f"\n📋 Local deployment steps:")
    print(f"   1. Download {gguf_files[0]} from Google Drive")
    print(f"   2. Rename to mits-tutor-qwen3-4b-q8_0.gguf")
    print(f"   3. Place next to training/Modelfile")
    print(f"   4. Run: ollama create mits-tutor -f training/Modelfile")
    print(f"   5. Run: ollama run mits-tutor")
else:
    print("❌ No GGUF file found — check errors above")